<a href="https://colab.research.google.com/github/muterae/PS-Agents/blob/main/UCS_Trailer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Uniform Cost Search: Self-Driving Trailer, Nairobi to Kisumu


## Problem

| Element | Definition |
|---|---|
| **Initial state** | The trailer is in Nairobi |
| **Goal** | The trailer is in Kisumu |
| **Actions** | Drive along a road from the current town to a neighbouring town |
| **Transition** | Driving from town A to town B along a road puts the trailer in B |
| **Path cost** | Sum of the road distances in km along the path |

## How It works

1. Put the start node in the **frontier** with cost 0.
2. Remove the frontier node with the **lowest path cost**.
3. Goal test on removal: if it is the goal, stop and return its path. (The test is done when a node is removed, not when it is generated, because a cheaper route to the goal may still be on the frontier.)
4. Otherwise add its town to the `explored-set` and generate a child for every road leaving it.
5. A child is added to the frontier if its town is not explored and not already on the frontier. If the town is on the frontier with a higher cost, that entry is replaced by the cheaper path. Otherwise the child is discarded.
6. Repeat from step 2.

## 1. The directed graph (data from Part 1)

> **Replace the distances below with your group's Google Maps values from Part 1.**

Each road is listed once as `(from, to, distance_km, road)`. A two-way road is stored as **two directed arcs**, one in each direction, with the same cost.

In [ ]:
towns = [
    'Nairobi', 'Kiambu', 'Limuru', 'Machakos', 'Thika', "Murang'a", 'Karatina', 'Nyeri', 'Nanyuki', 'Embu', 'Nyahururu', 'Naivasha', 'Gilgil', 'Nakuru', 'Molo', 'Narok', 'Bomet', 'Kericho', 'Kapsabet', 'Eldoret', 'Kitale', 'Kisumu', 'Kisii', 'Homa Bay', 'Siaya', 'Kakamega', 'Bungoma', 'Busia'
]

In [ ]:
# (from, to, distance_km, road)
roads = [
    ('Nairobi', 'Kiambu', 14, 'A2/Kiambu Rd'),
    ('Nairobi', 'Limuru', 30, 'A104'),
    ('Nairobi', 'Thika', 42, 'A2/Thika Superhighway'),
    ('Nairobi', 'Machakos', 64, 'A109 / Machakos Rd'),
    ('Kiambu', 'Limuru', 20, 'Kiambu-Limuru Rd'),
    ('Kiambu', 'Thika', 35, 'Kiambu-Ruiru-Thika'),
    ('Thika', 'Machakos', 50, 'Thika-Kangundo-Machakos'),
    ('Thika', "Murang'a", 40, 'B-road via Makuyu'),
    ('Thika', 'Nyeri', 100, 'A2'),
    ('Thika', 'Embu', 100, 'Thika-Kenol-Embu'),
    ("Murang'a", 'Karatina', 45, 'B-road'),
    ('Karatina', 'Nyeri', 25, 'A2'),
    ('Karatina', 'Embu', 55, 'B-road via Kerugoya'),
    ('Nyeri', 'Nanyuki', 60, 'A2'),
    ('Nyeri', 'Nyahururu', 80, 'B-road'),
    ('Limuru', 'Naivasha', 60, 'A104'),
    ('Naivasha', 'Gilgil', 30, 'A104'),
    ('Gilgil', 'Nakuru', 35, 'A104'),
    ('Nakuru', 'Nyahururu', 75, 'B-road'),
    ('Nakuru', 'Molo', 45, 'A104'),
    ('Nairobi', 'Narok', 140, 'A104/B3 via Suswa'),
    ('Naivasha', 'Narok', 105, 'Naivasha-Mai Mahiu-Narok'),
    ('Narok', 'Bomet', 85, 'B-road'),
    ('Bomet', 'Kericho', 45, 'B-road'),
    ('Bomet', 'Kisii', 85, 'B-road'),
    ('Molo', 'Kericho', 70, 'A104/B1 via Londiani'),
    ('Molo', 'Eldoret', 110, 'A104 via Timboroa'),
    ('Kericho', 'Kisumu', 80, 'A1 via Muhoroni'),
    ('Eldoret', 'Kapsabet', 40, 'B-road'),
    ('Kapsabet', 'Kisumu', 75, 'B-road'),
    ('Eldoret', 'Kitale', 70, 'A104/B-road'),
    ('Eldoret', 'Kakamega', 105, 'A1/B-road'),
    ('Kisumu', 'Kisii', 110, 'A1/B-road'),
    ('Kisumu', 'Homa Bay', 100, 'B-road'),
    ('Homa Bay', 'Kisii', 65, 'B-road'),
    ('Kisumu', 'Siaya', 60, 'A1/B-road'),
    ('Siaya', 'Busia', 50, 'B-road'),
    ('Kisumu', 'Kakamega', 50, 'A1'),
    ('Kakamega', 'Bungoma', 50, 'A1 via Webuye'),
    ('Bungoma', 'Busia', 50, 'A1/B-road'),
    ('Bungoma', 'Kitale', 60, 'B-road'),
    ('Kisumu', 'Busia', 105, 'A1 via Kakamega-Mumias'),
]

In [ ]:
class Graph:
    """Directed graph: each town maps to the list of arcs leaving it."""

    def __init__(self):
        self.arcs = {}

    def add_town(self, name):
        self.arcs.setdefault(name, [])

    def add_road(self, a, b, km, road):
        if a not in self.arcs or b not in self.arcs:
            raise KeyError(f"Unknown town in road: {a} - {b}")
        # A two-way road is two directed arcs
        self.arcs[a].append({"to": b, "cost": km, "road": road})
        self.arcs[b].append({"to": a, "cost": km, "road": road})

    def actions(self, town):
        """The arcs (roads) the trailer can take from this town."""
        return self.arcs[town]


kenya = Graph()
for name in towns:
    kenya.add_town(name)
for a, b, km, road in roads:
    kenya.add_road(a, b, km, road)

print(f"{len(kenya.arcs)} towns, {sum(len(v) for v in kenya.arcs.values())} directed arcs")

28 towns, 84 directed arcs


In [ ]:
for town, arcs in kenya.arcs.items():
    print(f"{town:10} -> " + ", ".join(f"{a['to']} ({a['cost']})" for a in arcs))

Nairobi    -> Kiambu (14), Limuru (30), Thika (42), Machakos (64), Narok (140)
Kiambu     -> Nairobi (14), Limuru (20), Thika (35)
Limuru     -> Nairobi (30), Kiambu (20), Naivasha (60)
Machakos   -> Nairobi (64), Thika (50)
Thika      -> Nairobi (42), Kiambu (35), Machakos (50), Murang'a (40), Nyeri (100), Embu (100)
Murang'a   -> Thika (40), Karatina (45)
Karatina   -> Murang'a (45), Nyeri (25), Embu (55)
Nyeri      -> Thika (100), Karatina (25), Nanyuki (60), Nyahururu (80)
Nanyuki    -> Nyeri (60)
Embu       -> Thika (100), Karatina (55)
Nyahururu  -> Nyeri (80), Nakuru (75)
Naivasha   -> Limuru (60), Gilgil (30), Narok (105)
Gilgil     -> Naivasha (30), Nakuru (35)
Nakuru     -> Gilgil (35), Nyahururu (75), Molo (45)
Molo       -> Nakuru (45), Kericho (70), Eldoret (110)
Narok      -> Nairobi (140), Naivasha (105), Bomet (85)
Bomet      -> Narok (85), Kericho (45), Kisii (85)
Kericho    -> Bomet (45), Molo (70), Kisumu (80)
Kapsabet   -> Eldoret (40), Kisumu (75)
Eldoret    -> Mol

## 2. Search node

A node is one path to a town. It stores the town (state), the parent node, the road taken from the parent, and the total path cost.

*Note: Following parent links back to the root rebuilds the route.*

In [ ]:
class Node:
    def __init__(self, state, parent=None, road=None, path_cost=0):
        self.state = state          # town
        self.parent = parent        # node we came from
        self.road = road            # road taken from the parent
        self.path_cost = path_cost  # total km from Nairobi along this path

    def route(self):
        """Towns from the start to this node."""
        towns_on_route, node = [], self
        while node is not None:
            towns_on_route.append(node.state)
            node = node.parent
        return towns_on_route[::-1]

    def legs(self):
        """(from, to, road, leg_km, cumulative_km) for each road driven."""
        chain, node = [], self
        while node.parent is not None:
            chain.append(node)
            node = node.parent
        return [
            (n.parent.state, n.state, n.road, n.path_cost - n.parent.path_cost, n.path_cost)
            for n in reversed(chain)
        ]

    def __repr__(self):
        return f"{self.state}({self.path_cost})"

## 3. Uniform Cost Search

The frontier is a dictionary of `town -> (node, arrival order)`. At each step the lowest-cost node is removed; ties are broken by which node arrived on the frontier first. With `verbose=True` the function prints each step: the frontier, the node removed, the goal test, how each child was handled, and the explored set.

In [ ]:
def show(frontier):
    items = sorted(frontier.values(), key=lambda item: (item[0].path_cost, item[1]))
    return "[" + ", ".join(f"{n.state}({n.path_cost})" for n, _ in items) + "]"


def uniform_cost_search(graph, start, goal, verbose=True):
    root = Node(start)
    frontier = {start: (root, 0)}      # town -> (node, arrival order)
    arrival = 1
    explored = []                      # towns in the order they were expanded
    step = 0

    while frontier:
        step += 1
        if verbose:
            print(f"Step {step}")
            print(f"  Frontier : {show(frontier)}")

        # Remove the lowest-cost node (ties: earliest arrival)
        town = min(frontier, key=lambda s: (frontier[s][0].path_cost, frontier[s][1]))
        node, _ = frontier.pop(town)

        # Goal test when the node is removed
        if node.state == goal:
            if verbose:
                print(f"  Remove   : {node.state} (cost {node.path_cost})  -> GOAL REACHED")
            return node, explored, step

        explored.append(node.state)
        if verbose:
            print(f"  Remove   : {node.state} (cost {node.path_cost})  -> not the goal, expand it")

        # Generate children
        for arc in graph.actions(node.state):
            child_town = arc["to"]
            cost = node.path_cost + arc["cost"]

            if child_town in explored:
                outcome = "skip (already explored)"
            elif child_town in frontier:
                old = frontier[child_town][0].path_cost
                if cost < old:
                    frontier[child_town] = (Node(child_town, node, arc["road"], cost), arrival)
                    arrival += 1
                    outcome = f"replace frontier entry ({cost} < {old})"
                else:
                    outcome = f"skip (frontier already has {old} <= {cost})"
            else:
                frontier[child_town] = (Node(child_town, node, arc["road"], cost), arrival)
                arrival += 1
                outcome = "add to frontier"

            if verbose:
                print(f"    {node.state} -> {child_town:10} cost {cost:4}  {outcome}")

        if verbose:
            print(f"  Explored : {explored}")
            print()

    return None, explored, step   # no route

## 4. Run the search: Nairobi to Kisumu

In [ ]:
goal_node, explored, steps = uniform_cost_search(kenya, "Nairobi", "Kisumu")

Step 1
  Frontier : [Nairobi(0)]
  Remove   : Nairobi (cost 0)  -> not the goal, expand it
    Nairobi -> Kiambu     cost   14  add to frontier
    Nairobi -> Limuru     cost   30  add to frontier
    Nairobi -> Thika      cost   42  add to frontier
    Nairobi -> Machakos   cost   64  add to frontier
    Nairobi -> Narok      cost  140  add to frontier
  Explored : ['Nairobi']

Step 2
  Frontier : [Kiambu(14), Limuru(30), Thika(42), Machakos(64), Narok(140)]
  Remove   : Kiambu (cost 14)  -> not the goal, expand it
    Kiambu -> Nairobi    cost   28  skip (already explored)
    Kiambu -> Limuru     cost   34  skip (frontier already has 30 <= 34)
    Kiambu -> Thika      cost   49  skip (frontier already has 42 <= 49)
  Explored : ['Nairobi', 'Kiambu']

Step 3
  Frontier : [Limuru(30), Thika(42), Machakos(64), Narok(140)]
  Remove   : Limuru (cost 30)  -> not the goal, expand it
    Limuru -> Nairobi    cost   60  skip (already explored)
    Limuru -> Kiambu     cost   50  skip (alread

## 5. Result: optimal path and its cost

In [ ]:
print("Optimal path :", " -> ".join(goal_node.route()))
print("Total cost   :", goal_node.path_cost, "km")
print("Towns expanded before the goal was reached:", len(explored))
print()
print(f"{'From':10} {'To':10} {'Road':28} {'Leg km':>7} {'Cumulative km':>14}")
for a, b, road, leg, total in goal_node.legs():
    print(f"{a:10} {b:10} {road:28} {leg:7} {total:14}")

Optimal path : Nairobi -> Limuru -> Naivasha -> Gilgil -> Nakuru -> Molo -> Kericho -> Kisumu
Total cost   : 350 km
Towns expanded before the goal was reached: 20

From       To         Road                          Leg km  Cumulative km
Nairobi    Limuru     A104                              30             30
Limuru     Naivasha   A104                              60             90
Naivasha   Gilgil     A104                              30            120
Gilgil     Nakuru     A104                              35            155
Nakuru     Molo       A104                              45            200
Molo       Kericho    A104/B1 via Londiani              70            270
Kericho    Kisumu     A1 via Muhoroni                   80            350


## 6. Conclusion

UCS always removes the cheapest path on the frontier, so the first time the goal is removed the path to it is optimal, provided every road cost is positive. The optimal path and its total cost are shown in section 5.